In [1]:
import requests
import pandas as pd

In [20]:
DEPARTMENTS = [
    "AFR",
    "ANT",
    "ARB",
    "ART",
    "BIO",
    "CHE",
    "CHI",
    "CLA",
    "COM",
    "CSC",
    "CIS",
    "DAN",
    "IDAT",
    "EAS",
    "ECO",
    "EDU",
    "ENG",
    "ENV",
    "XPL",
    "FMD",
    "FRE",
    "GSS",
    "IGEN",
    "GER",
    "LIT",
    "GRE",
    "SPA",
    "HIS",
    "HUM",
    "LAT",
    "LAS",
    "MAT",
    "IMES",
    "MIL",
    "MUS",
    "PHI",
    "PPE",
    "PHY",
    "POL",
    "PSY",
    "PBH",
    "REL",
    "RUS",
    "SOC",
    "SOU",
    "SIL",
    "THE",
    "WRI",
]

In [21]:
API_URL = "https://api.davidson.edu/api/public/v2/courses"
TERM_CODE = 202502
LIMIT = 50

In [22]:
def fetch_department_courses(department):
    all_courses = []
    offset = 0

    while True:
        params = {
            "departments": department,
            "term_code": TERM_CODE,
            "limit": LIMIT,
            "offset": offset
        }

        r = requests.get(API_URL, params=params)
        r.raise_for_status()
        data = r.json()

        if not data:
            break

        all_courses.extend(data)
        offset += LIMIT

    return all_courses

In [23]:
rows = []

for dept in DEPARTMENTS:
    courses = fetch_department_courses(dept)

    for course in courses:
        course_title = course["course_title"]

        meetings = course.get("meetings", [])

        for meeting in meetings:
            rows.append({
                "department": dept,
                "course_title": course_title,
                "meeting_type": meeting.get("type"),
                "weekdays": meeting.get("weekdays"),
                "start_time": meeting.get("start_time"),
                "end_time": meeting.get("end_time"),
                "class_time": meeting.get("class_time"),
                "room": meeting.get("room"),
                "building": 
                    (meeting["building"]["description"]
                    if meeting.get("building") is not None
                    else None),
            })

df = pd.DataFrame(rows)

df.head()

,department,course_title,meeting_type,weekdays,start_time,end_time,class_time,room,building
0,AFR,Intro to Africana Studies,lecture,MWF,1530,1620,0330 - 0420pm,2187,Chambers Building
1,AFR,Research Methods,lecture,TR,0940,1055,0940 - 1055am,1086,Chambers Building
2,AFR,"Race, Policing, and Justice",lecture,TR,1340,1455,0140 - 0255pm,109,Cunningham Fine Arts
3,AFR,Race and Campus Histories,lecture,TR,1215,1330,1215 - 0130pm,1006,Chambers Building
4,AFR,"Race, Gender, and Tourism",lecture,MW,1430,1545,0230 - 0345pm,3209,Chambers Building


In [ ]:
df.to_csv("davidson_courses_meetings.csv", index=False)